In [2]:
# Cell 1: Import necessary libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import logging
from lightgbm import LGBMRegressor
from lightgbm import early_stopping, log_evaluation

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 500)

In [3]:
# Cell 2: Load data
def load():
    logging.info("Loading data...")
    train = pd.read_csv(
        "train.csv", parse_dates=["date"]
    )
    test = pd.read_csv("test.csv", parse_dates=["date"])
    # sample_sub = pd.read_csv("sample_submission.csv")
    df = pd.concat([train, test], sort=False)
    logging.info("Data loaded successfully.")
    return df, train, test


df, train, test = load()

2024-08-26 23:25:31,722 - INFO - Loading data...
2024-08-26 23:25:31,945 - INFO - Data loaded successfully.


In [73]:
# Cell 3: Check dataframe
def check_df(dataframe, head=5):
    logging.info("Checking dataframe...")
    print("##################### Shape #####################")
    print(dataframe.shape)
    print("##################### Types #####################")
    print(dataframe.dtypes)
    print("##################### Unique #####################")
    print(dataframe.nunique())
    print("##################### Head #####################")
    print(dataframe.head(head))
    print("##################### Tail #####################")
    print(dataframe.tail(head))
    print("##################### NA #####################")
    print(dataframe.isnull().sum())
    logging.info("Dataframe check completed.")


check_df(df)

2024-08-26 22:55:10,486 - INFO - Checking dataframe...
2024-08-26 22:55:10,515 - INFO - Dataframe check completed.


##################### Shape #####################
(958000, 5)
##################### Types #####################
date     datetime64[ns]
store             int64
item              int64
sales           float64
id              float64
dtype: object
##################### Unique #####################
date      1916
store       10
item        50
sales      213
id       45000
dtype: int64
##################### Head #####################
        date  store  item  sales  id
0 2013-01-01      1     1   13.0 NaN
1 2013-01-02      1     1   11.0 NaN
2 2013-01-03      1     1   14.0 NaN
3 2013-01-04      1     1   13.0 NaN
4 2013-01-05      1     1   10.0 NaN
##################### Tail #####################
            date  store  item  sales       id
44995 2018-03-27     10    50    NaN  44995.0
44996 2018-03-28     10    50    NaN  44996.0
44997 2018-03-29     10    50    NaN  44997.0
44998 2018-03-30     10    50    NaN  44998.0
44999 2018-03-31     10    50    NaN  44999.0
###################

In [74]:
# Cell 4: Create date features
def create_date_features(df):
    logging.info("Creating date features...")
    df["month"] = df.date.dt.month
    df["day_of_month"] = df.date.dt.day
    df["day_of_year"] = df.date.dt.dayofyear
    df["week_of_year"] = df.date.dt.isocalendar().week
    df["day_of_week"] = df.date.dt.dayofweek
    df["year"] = df.date.dt.year
    df["is_wknd"] = df.date.dt.weekday // 4
    df["is_month_start"] = df.date.dt.is_month_start.astype(int)
    df["is_month_end"] = df.date.dt.is_month_end.astype(int)
    logging.info("Date features created.")
    return df


df = create_date_features(df)

2024-08-26 22:55:10,520 - INFO - Creating date features...
2024-08-26 22:55:10,661 - INFO - Date features created.


In [75]:
# Cell 5: Create lag features
def random_noise(dataframe):
    return np.random.normal(scale=1.6, size=(len(dataframe)))


def lag_features(dataframe, lags):
    logging.info("Creating lag features...")
    for lag in lags:
        dataframe["sales_lag_" + str(lag)] = dataframe.groupby(["store", "item"])[
            "sales"
        ].transform(lambda x: x.shift(lag)) + random_noise(dataframe)
    logging.info("Lag features created.")
    return dataframe


df = lag_features(df, [91, 98, 105, 112, 119, 126, 182, 364, 546, 728])

2024-08-26 22:55:10,666 - INFO - Creating lag features...
2024-08-26 22:55:11,858 - INFO - Lag features created.


In [76]:
# Cell 6: Create rolling mean features
def roll_mean_features(dataframe, windows):
    logging.info("Creating rolling mean features...")
    for window in windows:
        dataframe["sales_roll_mean_" + str(window)] = dataframe.groupby(
            ["store", "item"]
        )["sales"].transform(
            lambda x: x.shift(1)
            .rolling(window=window, min_periods=10, win_type="triang")
            .mean()
        ) + random_noise(
            dataframe
        )
    logging.info("Rolling mean features created.")
    return dataframe


df = roll_mean_features(df, [365, 546])

2024-08-26 22:55:11,864 - INFO - Creating rolling mean features...
2024-08-26 22:55:12,730 - INFO - Rolling mean features created.


In [77]:
# Cell 7: Create exponentially weighted mean features
def ewm_features(dataframe, alphas, lags):
    logging.info("Creating exponentially weighted mean features...")
    for alpha in alphas:
        for lag in lags:
            dataframe[
                "sales_ewm_alpha_" + str(alpha).replace(".", "") + "_lag_" + str(lag)
            ] = dataframe.groupby(["store", "item"])["sales"].transform(
                lambda x: x.shift(lag).ewm(alpha=alpha).mean()
            )
    logging.info("Exponentially weighted mean features created.")
    return dataframe


alphas = [0.95, 0.9, 0.8, 0.7, 0.5]
lags = [91, 98, 105, 112, 180, 270, 365, 546, 728]

df = ewm_features(df, alphas, lags)

2024-08-26 22:55:12,736 - INFO - Creating exponentially weighted mean features...
2024-08-26 22:55:18,868 - INFO - Exponentially weighted mean features created.


In [78]:
# Cell 8: One-hot encoding and log transformation
df = pd.get_dummies(
    df, columns=["store", "item", "day_of_week", "month"], drop_first=True, dtype=int
)

df["sales"] = np.log1p(df["sales"].values)

In [79]:
# Cell 9: Split data into train and validation sets
# Train set until the beginning of 2017 (end of 2016)
train = df.loc[(df["date"] < "2017-01-01"), :]

# Validation set for the first 3 months of 2017. Because the date range we need to forecast is the first 3 months of 2018.
val = df.loc[(df["date"] >= "2017-01-01") & (df["date"] < "2017-04-01"), :]

cols = [col for col in train.columns if col not in ["date", "id", "sales", "year"]]

y_train = train["sales"]
X_train = train[cols]

y_val = val["sales"]
X_val = val[cols]

In [80]:
# Cell 10: Define SMAPE metric
def smape(y_pred, y_true):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    n = len(y_true)
    masked_arr = ~((y_pred == 0) & (y_true == 0))
    y_pred, y_true = y_pred[masked_arr], y_true[masked_arr]
    num = np.abs(y_pred - y_true)
    denom = np.abs(y_pred) + np.abs(y_true)
    smape_val = (200 * np.sum(num / denom)) / n
    return smape_val


def lgbm_smape(y_pred, y_true):
    smape_val = smape(np.expm1(y_true), np.expm1(y_pred))
    return "SMAPE", smape_val, False

In [81]:
# Cell 11: Train LightGBM model
logging.info("Training LightGBM model...")
model = LGBMRegressor(
    num_leaves=10,
    learning_rate=0.01,
    max_depth=5,
    n_estimators=10000,
    n_jobs=-1,
    random_state=44,
    force_col_wise=True,
)

es = early_stopping(stopping_rounds=200, verbose=True)
le = log_evaluation(period=100)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_metric=lgbm_smape,
    callbacks=[es, le],
)

best_iteration = model.best_iteration_
y_pred_val = model.predict(X_val)

smape_score = smape(np.expm1(y_pred_val), np.expm1(y_val))
logging.info(f"Validation SMAPE score: {smape_score}")

2024-08-26 22:55:20,962 - INFO - Training LightGBM model...


[LightGBM] [Info] Total Bins 15025
[LightGBM] [Info] Number of data points in the train set: 730500, number of used features: 138
[LightGBM] [Info] Start training from score 3.789881
Training until validation scores don't improve for 200 rounds
[100]	training's l2: 0.0957918	training's SMAPE: 24.3843	valid_1's l2: 0.0968163	valid_1's SMAPE: 23.9963
[200]	training's l2: 0.0520664	training's SMAPE: 17.7413	valid_1's l2: 0.0536927	valid_1's SMAPE: 17.563
[300]	training's l2: 0.0406797	training's SMAPE: 15.6678	valid_1's l2: 0.0424164	valid_1's SMAPE: 15.8074
[400]	training's l2: 0.0360684	training's SMAPE: 14.7543	valid_1's l2: 0.0377553	valid_1's SMAPE: 15.0714
[500]	training's l2: 0.0339373	training's SMAPE: 14.3241	valid_1's l2: 0.0356413	valid_1's SMAPE: 14.7333
[600]	training's l2: 0.0328619	training's SMAPE: 14.1088	valid_1's l2: 0.0346772	valid_1's SMAPE: 14.5822
[700]	training's l2: 0.0322047	training's SMAPE: 13.9763	valid_1's l2: 0.0341203	valid_1's SMAPE: 14.4859
[800]	training

2024-08-26 23:03:49,677 - INFO - Validation SMAPE score: 13.570466480744322


In [82]:
# Cell 12: Plot feature importance
def plot_importance(model, features):
    logging.info("Plotting feature importance...")
    feature_imp = pd.DataFrame(
        {"Value": model.feature_importances_, "Feature": features.columns}
    )
    feature_imp = feature_imp.sort_values(by="Value", ascending=False)
    print(len(feature_imp))
    return feature_imp


feat_imp = plot_importance(model, X_train)

importance_low = feat_imp[feat_imp["Value"] < 50]["Feature"].values

imp_feats = [col for col in cols if col not in importance_low]

train = df.loc[~df.sales.isna()]
y_train = train["sales"]
X_train = train[imp_feats]
print(best_iteration)
test = df.loc[df.sales.isna()]
X_test = test[imp_feats]

2024-08-26 23:03:49,703 - INFO - Plotting feature importance...


138
10000


In [83]:
# Cell 13: Train final LightGBM model

logging.info("Training final LightGBM model...")
final_model = LGBMRegressor(
    num_leaves=10,
    learning_rate=0.01,
    max_depth=5,
    n_estimators=best_iteration,
    n_jobs=-1,
    random_state=44,
    force_col_wise=True,
)
final_model.fit(X_train, y_train)
test_preds = final_model.predict(X_test)
test_preds = np.expm1(test_preds)
logging.info("Final model training completed.")

model_filename = "final_model.txt"

final_model.booster_.save_model(model_filename)
logging.info(f"Final model saved to {model_filename}.")

2024-08-26 23:03:51,415 - INFO - Training final LightGBM model...


[LightGBM] [Info] Total Bins 14995
[LightGBM] [Info] Number of data points in the train set: 913000, number of used features: 123
[LightGBM] [Info] Start training from score 3.820443


2024-08-26 23:09:38,354 - INFO - Final model training completed.
2024-08-26 23:09:38,480 - INFO - Final model saved to final_model.txt.


In [84]:
# Cell 14: Load data again and create forecast dataframe
df_, train_, test_ = load()
forecast = pd.DataFrame(
    {
        "date": test_["date"],
        "store": test_["store"],
        "item": test_["item"],
        "sales": test_preds,
    }
)

2024-08-26 23:09:38,501 - INFO - Loading data...
2024-08-26 23:09:38,781 - INFO - Data loaded successfully.


In [87]:
# Cell 15: Define function to plot predicted graph
def predicted_graph(store_num, item_num, train, forecast):
    logging.info(f"Plotting predicted graph for store {store_num}, item {item_num}...")
    train_data = train[(train.store == store_num) & (train.item == item_num)]
    forecast_data = forecast[
        (forecast.store == store_num) & (forecast.item == item_num)
    ]

    fig = make_subplots(rows=1, cols=1, shared_xaxes=True)

    fig.add_trace(
        go.Scatter(
            x=train_data.date,
            y=train_data.sales,
            name=f"Store {store_num} Item {item_num} Sales",
            mode="lines",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=forecast_data.date,
            y=forecast_data.sales,
            name=f"Store {store_num} Item {item_num} Forecast",
            mode="lines",
        )
    )

    fig.update_layout(
        title=f"Sales and Forecast for Store {store_num}, Item {item_num}",
        xaxis_title="Date",
        yaxis_title="Sales",
        legend_title="Legend",
        hovermode="x unified",
    )

    fig.update_xaxes(
        rangeslider_visible=True,
        rangeselector=dict(
            buttons=list(
                [
                    dict(count=1, label="1m", step="month", stepmode="backward"),
                    dict(count=6, label="6m", step="month", stepmode="backward"),
                    dict(count=1, label="YTD", step="year", stepmode="todate"),
                    dict(count=1, label="1y", step="year", stepmode="backward"),
                    dict(step="all"),
                ]
            )
        ),
    )

    fig.show()
    logging.info(f"Predicted graph for store {store_num}, item {item_num} plotted.")